In [1]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA


data = pd.read_csv('data/application_train_FE_baked.csv')

In [13]:
# New Features

def add_engineered_features(df, target="TARGET"):
    df = df.copy()
    eps = 1e-6

    # ---- Ratios / capacity ----
    df["credit_to_income"]  = df["AMT_CREDIT"]  / (df["AMT_INCOME_TOTAL"] + eps)
    df["annuity_to_income"] = df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"] + eps)
    df["annuity_to_credit"] = df["AMT_ANNUITY"] / (df["AMT_CREDIT"] + eps)
    df["income_per_member"] = df["AMT_INCOME_TOTAL"] / np.maximum(df["CNT_FAM_MEMBERS"], 1)
    df["income_per_child"]  = df["AMT_INCOME_TOTAL"] / (1 + np.maximum(df["CNT_CHILDREN"], 0))
    df["dsr_monthly"]       = df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"]/12.0 + eps)

    # ---- Time conversions & contactability ----
    df["age_years"]    = -df["DAYS_BIRTH"] / 365.25
    df["reg_years"]    = -df["DAYS_REGISTRATION"] / 365.25
    df["id_years"]     = -df["DAYS_ID_PUBLISH"] / 365.25
    df["phone_years"]  = -df["DAYS_LAST_PHONE_CHANGE"] / 365.25
    df["contactability"] = df["FLAG_PHONE"] + df["FLAG_EMAIL"]

    # ---- Previous behavior & pricing ----
    df["prev_interest_per_credit"] = df["prev_interest_mean"] / (df["prev_amt_credit_mean"] + eps)
    df["prev_payments_ratio"]      = df["prev_last3_cnt_payment_mean"] / (df["prev_cnt_payment_mean"] + eps)
    df["recent_approval_momentum"] = df["prev_last3_approved_rate"] - df["prev_last5_approved_rate"]
    df["recent_credit_growth"]     = df["prev_last3_amt_credit_mean"] - df["prev_last5_amt_credit_mean"]
    df["prev_rate_spread"]         = df["prev_rate_mean_all"] - df["prev_rate_med_all"]

    # ---- Region transforms ----
    # If inputs may be standardized/negative, clip to keep log1p well-defined
    df["region_pop_log"]      = np.log1p(np.clip(df["REGION_POPULATION_RELATIVE"], 0, None))
    df["region_rating_x_pop"] = df["REGION_RATING"] * df["REGION_POPULATION_RELATIVE"]

    # ---- Magnitudes (log) & mild nonlinearity ----
    df["log_income"]  = np.log1p(np.clip(df["AMT_INCOME_TOTAL"], 0, None))
    df["log_credit"]  = np.log1p(np.clip(df["AMT_CREDIT"],       0, None))
    df["log_annuity"] = np.log1p(np.clip(df["AMT_ANNUITY"],      0, None))
    df["ext2_sq"]     = df["EXT_SOURCE_2"] ** 2

    # ---- Hygiene: replace inf -> NaN, then fill numerics (binary->0, continuous->median) ----
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    num_cols_no_t = [c for c in num_cols if c != target]

    bin_cols, cont_cols = [], []
    for c in num_cols_no_t:
        vals = pd.unique(df[c].dropna())
        if len(vals) <= 3 and set(vals).issubset({0, 1}):
            bin_cols.append(c)
        else:
            cont_cols.append(c)

    if bin_cols:
        df[bin_cols] = df[bin_cols].fillna(0)
    if cont_cols:
        df[cont_cols] = df[cont_cols].fillna(df[cont_cols].median())

    # Ensure target dtype if present; optional downcast for speed
    if target in df.columns:
        df[target] = df[target].astype(int)

    float_cols = [c for c in df.columns if c != target and pd.api.types.is_float_dtype(df[c])]
    df[float_cols] = df[float_cols].astype("float32")

    return df

data_fe = add_engineered_features(data)

In [14]:
# Make random split, stratified split by target variable, and stratified split by region rating, debt ratio, and target
train_data_random, val_data_random = train_test_split(data_fe, test_size=0.2, random_state=42)
train_data_stratified, val_data_stratified = train_test_split(data_fe, test_size=0.2, random_state=42, stratify=data_fe['TARGET'])

# # For custom stratification, create bins for continuous variables first
# data['REGION_RATING_bins'] = pd.qcut(data['REGION_RATING'], q=4, labels=['VL', 'L', 'H', 'VH'], duplicates='drop')
# data['debt_ratio_bins'] = pd.qcut(data['debt_ratio'], q=4, labels=['VL', 'L', 'H', 'VH'], duplicates='drop')

# # Create a combined stratification column
# data['strat_col'] = data['REGION_RATING_bins'].astype(str) + '_' + data['debt_ratio_bins'].astype(str) + '_' + data['TARGET'].astype(str)

# # Now stratify by the combined column
# train_data_custom, val_data_custom = train_test_split(data, test_size=0.2, random_state=42, stratify=data['strat_col'])

In [15]:
# Metrics
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def recall(y_true, y_pred):
    true_positives = np.sum((y_true == 1) & (y_pred == 1))
    possible_positives = np.sum(y_true == 1)
    return true_positives / possible_positives if possible_positives > 0 else 0

def precision(y_true, y_pred):
    true_positives = np.sum((y_true == 1) & (y_pred == 1))
    predicted_positives = np.sum(y_pred == 1)
    return true_positives / predicted_positives if predicted_positives > 0 else 0

def specificity(y_true, y_pred):
    true_negatives = np.sum((y_true == 0) & (y_pred == 0))
    possible_negatives = np.sum(y_true == 0)
    return true_negatives / possible_negatives if possible_negatives > 0 else 0

def f1_score(y_true, y_pred):
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    return 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0

def roc_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    P = np.sum(y_true == 1)
    N = np.sum(y_true == 0)
    if P == 0 or N == 0:
        return 0.5
    order = np.argsort(-y_score, kind='mergesort')
    y_sorted = y_true[order]
    s_sorted = y_score[order]
    tp = np.cumsum(y_sorted)
    fp = np.cumsum(1 - y_sorted)
    changes = np.where(np.diff(s_sorted) != 0)[0]
    cut_idx = np.r_[changes, len(s_sorted) - 1]
    tpr = tp[cut_idx] / P
    fpr = fp[cut_idx] / N
    tpr = np.r_[0.0, tpr, 1.0]
    fpr = np.r_[0.0, fpr, 1.0]

    return float(np.trapz(tpr, fpr))

In [16]:
# Cross-validation
def cross_validate(model, train_data, val_data, cv=5):
    X = train_data.drop(columns=['TARGET']).values
    y = train_data['TARGET'].values
    val_X = val_data.drop(columns=['TARGET']).values
    val_y = val_data['TARGET'].values


    fold_size = len(X) // cv
    metrics = {'accuracy': [], 'recall': [], 'precision': [], 'specificity': [], 'f1_score': [], 'roc_auc': []}
    
    for fold in range(cv):
        start = fold * fold_size
        end = (fold + 1) * fold_size if fold != cv - 1 else len(X)
        
        X_val_fold = X[start:end]
        y_val_fold = y[start:end]
        X_train_fold = np.concatenate([X[:start], X[end:]], axis=0)
        y_train_fold = np.concatenate([y[:start], y[end:]], axis=0)
        
        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)
        if hasattr(model, "predict_proba"):
            y_pred_prob = model.predict_proba(X_val_fold)[:, 1]
        else:
            y_pred_prob = model.decision_function(X_val_fold)
            y_pred_prob = (y_pred_prob - y_pred_prob.min()) / (y_pred_prob.max() - y_pred_prob.min())
        

        metrics['accuracy'].append(accuracy(y_val_fold, y_pred))
        metrics['recall'].append(recall(y_val_fold, y_pred))
        metrics['precision'].append(precision(y_val_fold, y_pred))
        metrics['specificity'].append(specificity(y_val_fold, y_pred))
        metrics['f1_score'].append(f1_score(y_val_fold, y_pred))
        metrics['roc_auc'].append(roc_auc(y_val_fold, y_pred_prob))

    # Evaluate on validation set
    model.fit(X, y)
    y_val_pred = model.predict(val_X)
    if hasattr(model, "predict_proba"):
        y_val_pred_prob = model.predict_proba(val_X)[:, 1]
    else:
        y_val_pred_prob = model.decision_function(val_X)
        y_val_pred_prob = (y_val_pred_prob - y_val_pred_prob.min()) / (y_val_pred_prob.max() - y_val_pred_prob.min())

    val_metrics = {
        'accuracy': accuracy(val_y, y_val_pred),
        'recall': recall(val_y, y_val_pred),
        'precision': precision(val_y, y_val_pred),
        'specificity': specificity(val_y, y_val_pred),
        'f1_score': f1_score(val_y, y_val_pred),
        'roc_auc': roc_auc(val_y, y_val_pred_prob)
    }
    
    # Calculate average coefficients accross folds if applicable
    coefs = None
    if hasattr(model, 'coef_'):
        coefs = model.coef_
    avg_metrics = {key: np.mean(value) for key, value in metrics.items()}
    return avg_metrics, coefs, val_metrics

In [25]:
# Models
# Logistic Regression - set class weights to reflect class imbalance

class_w = {0: 1, 1: 18}
log_reg = LogisticRegression(max_iter=1000, class_weight=class_w)

# SVC
svm_model = LinearSVC(
    class_weight=class_w,
    C=1.0,
    tol=1e-3,
    max_iter=5000,
    dual='auto'
)

# LDA
lda_model = LDA(
    solver='lsqr',
    shrinkage='auto',
    priors = [0.25, 0.75]
)


In [26]:
# Run Models
# Define feature sets for different models
target = "TARGET"

# Basic demographic and financial features
predictors1 = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "debt_ratio",
    "CNT_CHILDREN", "CNT_FAM_MEMBERS", "REGION_POPULATION_RELATIVE",
    "REGION_RATING", "DAYS_BIRTH", "LIVE_CITY_NOT_WORK_CITY"
]

# Previous application statistics
predictors2 = [
    "prev_app_count", "prev_approved_rate", "prev_refused_rate",
    "prev_amt_credit_mean", "prev_cnt_payment_mean", "prev_ann_to_credit_mean",
    "prev_interest_mean", "prev_rate_mean_all", "prev_share_mean_all"
]

# Recent application history features
predictors3 = [
    "prev_last3_n", "prev_last3_approved_rate", "prev_last3_amt_credit_mean",
    "prev_last3_cnt_payment_mean", "prev_last3_ann_to_credit_mean",
    "prev_last5_n", "prev_last5_approved_rate", "prev_last5_cnt_payment_mean"
]

# Document and registration features
predictors4 = [
    "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE",
    "FLAG_PHONE", "FLAG_EMAIL", "LIVE_CITY_NOT_WORK_CITY",
    "REGION_POPULATION_RELATIVE", "REGION_RATING"
]

# Contract and property features
predictors5 = [
    "NAME_CONTRACT_TYPE_Cash.loans", "NAME_CONTRACT_TYPE_Revolving.loans",
    "FLAG_OWN_REALTY_Y", "FLAG_OWN_CAR_Y",
    "NAME_HOUSING_TYPE_House...apartment", "NAME_HOUSING_TYPE_Rented.apartment",
    "NAME_HOUSING_TYPE_With.parents", "AMT_CREDIT", "AMT_ANNUITY"
]

# Income and education features
predictors6 = [
    "NAME_INCOME_TYPE_Working", "NAME_INCOME_TYPE_Commercial.associate",
    "NAME_INCOME_TYPE_Pensioner", "NAME_INCOME_TYPE_State.servant",
    "NAME_EDUCATION_TYPE_Secondary...secondary.special",
    "NAME_EDUCATION_TYPE_Higher.education", "NAME_EDUCATION_TYPE_Lower.secondary",
    "AMT_INCOME_TOTAL", "REGION_RATING"
]

# Combined important features
predictors7 = [
    "EXT_SOURCE_2", "AMT_CREDIT", "AMT_ANNUITY", "AMT_INCOME_TOTAL",
    "debt_ratio", "prev_approved_rate", "prev_app_count",
    "DAYS_BIRTH", "DAYS_REGISTRATION", "REGION_RATING"
]

# Most important features subset
predictors8 = [
    "EXT_SOURCE_2", "AMT_CREDIT", "AMT_INCOME_TOTAL", "debt_ratio", "prev_approved_rate"
]

feature_sets = {
    'Model_1': predictors1,
    'Model_2': predictors2,
    'Model_3': predictors3,
    'Model_4': predictors4,
    'Model_5': predictors5,
    'Model_6': predictors6,
    'Model_7': predictors7,
    'Model_8': predictors8
}

# Model 1
train_df = train_data_random[feature_sets['Model_1'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_1'] + [target]].copy()
log_reg_metrics1 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics1

# Model 2
train_df = train_data_random[feature_sets['Model_2'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_2'] + [target]].copy()
log_reg_metrics2 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics2

# Model 3
train_df = train_data_random[feature_sets['Model_3'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_3'] + [target]].copy()
log_reg_metrics3 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics3

# Model 4
train_df = train_data_random[feature_sets['Model_4'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_4'] + [target]].copy()
log_reg_metrics4 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics4

# Model 5
train_df = train_data_random[feature_sets['Model_5'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_5'] + [target]].copy()
log_reg_metrics5 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics5

# Model 6
train_df = train_data_random[feature_sets['Model_6'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_6'] + [target]].copy()
log_reg_metrics6 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics6

# Model 7
train_df = train_data_random[feature_sets['Model_7'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_7'] + [target]].copy()
log_reg_metrics7 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics7

# Model 8
train_df = train_data_random[feature_sets['Model_8'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_8'] + [target]].copy()
log_reg_metrics8 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics8



({'accuracy': 0.41748818151458095,
  'recall': 0.8278483900155921,
  'precision': 0.1054477703259167,
  'specificity': 0.3813475920169062,
  'f1_score': 0.18705918179009542,
  'roc_auc': 0.6755383511619067},
 array([[-0.52444975, -0.0398887 ,  0.00133421,  0.31817903, -0.19363646]]),
 {'accuracy': 0.4151186286012254,
  'recall': 0.820983874260053,
  'precision': 0.10302781904810697,
  'specificity': 0.3799075599001222,
  'f1_score': 0.18308031954844434,
  'roc_auc': 0.6770581594062391})

In [27]:
print(log_reg_metrics1)
print(log_reg_metrics2)
print(log_reg_metrics3)
print(log_reg_metrics4)
print(log_reg_metrics5)
print(log_reg_metrics6)
print(log_reg_metrics7)
print(log_reg_metrics8)

train_df = train_data_random[feature_sets['Model_8'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_8'] + [target]].copy()
svc_metrics1 = cross_validate(svm_model, train_df, val_df, cv=5)
print("SVM Metrics:", svc_metrics1)

train_df = train_data_random[feature_sets['Model_8'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_8'] + [target]].copy()
lda_metrics1 = cross_validate(lda_model, train_df, val_df, cv=5)
print("LDA Metrics:", lda_metrics1)

({'accuracy': 0.271131030681805, 'recall': 0.8868296339992521, 'precision': 0.0907102961154746, 'specificity': 0.21691075107441313, 'f1_score': 0.1645777161855942, 'roc_auc': 0.6264666468786023}, array([[ 6.37844458e-05, -1.49225766e-01,  1.07330010e-01,
         3.02255203e-01,  7.53779160e-02, -1.14587826e-01,
        -2.82960399e-02,  2.01480411e-01,  2.66831265e-01,
         8.09586267e-02]]), {'accuracy': 0.2682342588971451, 'recall': 0.8856909573382323, 'precision': 0.08912212956496735, 'specificity': 0.21466645416068994, 'f1_score': 0.16194830642903799, 'roc_auc': 0.6197976796919623})
({'accuracy': 0.20540750222561793, 'recall': 0.9221150884176484, 'precision': 0.08651597553622417, 'specificity': 0.14227845630945832, 'f1_score': 0.15818403680857562, 'roc_auc': 0.6168888861304774}, array([[-0.10061511, -0.05259916,  0.23583669, -0.24029717,  0.05781405,
        -0.11231276,  0.06639259,  0.18249009,  0.08290204]]), {'accuracy': 0.20165232694563942, 'recall': 0.9228413962033069, '

In [28]:
predictors_AUC_7pp = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","AMT_INCOME_TOTAL","debt_ratio",
    "prev_approved_rate","prev_app_count","DAYS_BIRTH","DAYS_REGISTRATION","REGION_RATING",
    "credit_to_income","annuity_to_credit","age_years","reg_years",
    "prev_rate_spread","prev_interest_per_credit","ext2_sq"
]

# 2) Recent-behavior heavy (recency + momentum + a few capacity anchors)
predictors_RECENT = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_INCOME_TOTAL","debt_ratio","age_years",
    "prev_last3_approved_rate","prev_last5_approved_rate","recent_approval_momentum",
    "prev_last3_cnt_payment_mean","prev_last5_cnt_payment_mean",
    "prev_interest_per_credit","prev_rate_spread","ext2_sq"
]

# 3) Affordability & capacity focus (clean signal for linear models)
predictors_CAPACITY = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","AMT_INCOME_TOTAL","debt_ratio",
    "credit_to_income","annuity_to_income","annuity_to_credit","income_per_member","dsr_monthly",
    "REGION_RATING","age_years","prev_approved_rate","ext2_sq"
]

# 4) Stability + contactability + region interaction (often improves ranking)
predictors_STABILITY = [
    "EXT_SOURCE_2","REGION_RATING","REGION_POPULATION_RELATIVE","region_rating_x_pop",
    "DAYS_BIRTH","reg_years","id_years","phone_years","FLAG_PHONE","FLAG_EMAIL",
    "AMT_INCOME_TOTAL","debt_ratio","prev_approved_rate","AMT_CREDIT","ext2_sq"
]

# 5) Tiny, high-signal (9 vars) for speed + surprisingly good AUC
predictors_TINY9 = [
    "EXT_SOURCE_2","debt_ratio","prev_approved_rate",
    "credit_to_income","annuity_to_credit","age_years",
    "prev_rate_spread","prev_interest_per_credit","REGION_RATING"
]

# 6) Contract/property tilt + core finance (diversifies signal a bit)
predictors_CONTRACT = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","debt_ratio",
    "NAME_CONTRACT_TYPE_Cash.loans","NAME_CONTRACT_TYPE_Revolving.loans","FLAG_OWN_REALTY_Y",
    "REGION_RATING","credit_to_income","annuity_to_credit","age_years","prev_approved_rate","ext2_sq"
]

# Model AUC 7pp
train_df = train_data_random[predictors_AUC_7pp + [target]].copy()
val_df   = val_data_random[predictors_AUC_7pp + [target]].copy()
log_reg_metrics9 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics9)

# Model RECENT
train_df = train_data_random[predictors_RECENT + [target]].copy()
val_df   = val_data_random[predictors_RECENT + [target]].copy()
log_reg_metrics10 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics10)

# Model Capacity
train_df = train_data_random[predictors_CAPACITY + [target]].copy()
val_df   = val_data_random[predictors_CAPACITY + [target]].copy()
log_reg_metrics11 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics11)

# Model Stability
train_df = train_data_random[predictors_STABILITY + [target]].copy()
val_df   = val_data_random[predictors_STABILITY + [target]].copy()
log_reg_metrics12 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics12)

# Model Tiny9
train_df = train_data_random[predictors_TINY9 + [target]].copy()
val_df   = val_data_random[predictors_TINY9 + [target]].copy()
log_reg_metrics13 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics13)

# Model Contract
train_df = train_data_random[predictors_CONTRACT + [target]].copy()
val_df   = val_data_random[predictors_CONTRACT + [target]].copy()
log_reg_metrics14 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics14)


({'accuracy': 0.4342072742319008, 'recall': 0.8308459953661884, 'precision': 0.10860091696901339, 'specificity': 0.3992783743943983, 'f1_score': 0.19208420943027177, 'roc_auc': 0.6887159686444277}, array([[-5.12931597e-01, -9.56897749e-02,  1.08730557e-01,
        -7.29522256e-04,  2.94205809e-01, -2.19531398e-01,
        -3.64191865e-02,  2.23875395e-01,  3.31536409e-02,
         7.39610367e-02, -4.08521865e-04, -1.23225060e-04,
        -6.12937424e-04, -9.07697188e-05, -5.43584201e-03,
        -7.28203006e-06, -2.57003776e-02]]), {'accuracy': 0.4321959327336723, 'recall': 0.8197591345172485, 'precision': 0.10574543156564327, 'specificity': 0.39857266818962617, 'f1_score': 0.18732653870373395, 'roc_auc': 0.6848182766822973})


c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

({'accuracy': 0.42371303933058907, 'recall': 0.8299694040301105, 'precision': 0.10670273092338176, 'specificity': 0.38793423552937734, 'f1_score': 0.18908698948925834, 'roc_auc': 0.6823880453114983}, array([[-5.49536133e-01, -3.29732844e-02,  5.78762161e-04,
         3.01299136e-01, -4.37545782e+01, -4.61307425e-02,
        -1.19125008e-01,  7.30347701e-02, -1.24621321e-02,
         5.93894569e-02, -7.16173251e-06, -4.77383509e-02,
        -3.53611516e-02]]), {'accuracy': 0.4228425237909008, 'recall': 0.8238416003265973, 'precision': 0.10458126036484246, 'specificity': 0.3880536223414617, 'f1_score': 0.18560161872571337, 'roc_auc': 0.6832633154913695})


c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

({'accuracy': 0.4204946875145181, 'recall': 0.829940904409435, 'precision': 0.10615577695952697, 'specificity': 0.3844365181689341, 'f1_score': 0.18822678617775607, 'roc_auc': 0.6781552893728516}, array([[-5.40132179e-01, -1.54298843e-01,  1.62849946e-01,
        -2.75940849e-01,  3.22486846e-01, -3.38044354e-04,
        -3.70839287e-02, -1.33493100e-04,  2.76264177e-01,
         3.01924416e-03,  7.25634544e-02, -3.08749562e-01,
        -1.91521060e-01, -3.23279006e-02]]), {'accuracy': 0.41695997914222394, 'recall': 0.8275158195550112, 'precision': 0.10397804508964066, 'specificity': 0.3813419752430537, 'f1_score': 0.18474298213634707, 'roc_auc': 0.6796482124557747})
({'accuracy': 0.4321214548215531, 'recall': 0.8308845023596666, 'precision': 0.10824211936874228, 'specificity': 0.3970039030584355, 'f1_score': 0.19152435474574644, 'roc_auc': 0.6878664800194549}, array([[-0.50584921,  0.07177868,  0.01858124,  0.01604477,  0.24564412,
        -0.00113097, -0.00280821, -0.00273728, -0.054

In [ ]:
predictors_COEF_CORE20 = [
    "EXT_SOURCE_2","debt_ratio","AMT_CREDIT","AMT_ANNUITY","AMT_INCOME_TOTAL",
    "credit_to_income","annuity_to_credit","annuity_to_income",
    "age_years","reg_years","REGION_RATING",
    "prev_approved_rate","prev_app_count",
    "prev_rate_spread","prev_interest_per_credit","prev_cnt_payment_mean",
    "prev_last3_approved_rate","prev_last5_approved_rate",
    "ext2_sq","region_rating_x_pop"
]

predictors_COEF_RECENT_CAP = [
    "EXT_SOURCE_2","debt_ratio","credit_to_income","annuity_to_credit","dsr_monthly","age_years",
    "prev_last3_approved_rate","prev_last5_approved_rate","recent_approval_momentum",
    "prev_last3_cnt_payment_mean","prev_last5_cnt_payment_mean",
    "prev_interest_per_credit","prev_rate_spread","recent_credit_growth",
    "prev_payments_ratio","REGION_RATING","ext2_sq"
]

predictors_COEF_TINY12 = [
    "EXT_SOURCE_2","debt_ratio","prev_approved_rate",
    "credit_to_income","annuity_to_credit","age_years",
    "prev_rate_spread","prev_interest_per_credit","REGION_RATING",
    "prev_last3_approved_rate","prev_app_count","ext2_sq"
]

predictors_COEF_NOISE_REDUCED = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","debt_ratio",
    "credit_to_income","annuity_to_credit","income_per_member",
    "age_years","REGION_RATING","prev_approved_rate","prev_app_count",
    "prev_last3_approved_rate","prev_rate_spread","prev_interest_per_credit",
    "region_rating_x_pop","ext2_sq"
]


feature_sets_new = {
    "COEF_CORE20": predictors_COEF_CORE20,
    "COEF_RECENT_CAP": predictors_COEF_RECENT_CAP,
    "COEF_TINY12": predictors_COEF_TINY12,
    "COEF_NOISE_REDUCED": predictors_COEF_NOISE_REDUCED,
}

for name, cols in feature_sets_new.items():
    train_df = train_data_random[cols + [target]].copy()
    val_df   = val_data_random[cols + [target]].copy()
    print(f"\n=== {name} ===")
    print("LR:",  cross_validate(log_reg, train_df, val_df, cv=5))
    print("SVM:", cross_validate(svm_model, train_df, val_df, cv=5))
    print("LDA:", cross_validate(lda_model, train_df, val_df, cv=5))



=== COEF_CORE20 ===


c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

LR: ({'accuracy': 0.42338708471825565, 'recall': 0.8292848945762484, 'precision': 0.1065811368984854, 'specificity': 0.387642412917025, 'f1_score': 0.1888770488564615, 'roc_auc': 0.6806205865190279}, array([[-5.37179128e-01,  3.19150574e-01, -1.64010905e-01,
         1.68827717e-01,  7.59795040e-05, -3.82548692e-04,
        -1.24875951e-04, -2.64725048e-05, -1.58963112e-01,
        -6.75134803e-02,  6.82647176e-02, -2.41571987e-01,
        -6.33421157e-02, -3.45342350e-02, -6.95745505e-06,
         6.45189246e-02,  3.61907950e-02,  2.31379824e-03,
        -2.97093071e-02,  1.83138874e-02]]), {'accuracy': 0.4207567461869378, 'recall': 0.8240457236170647, 'precision': 0.10425597851350653, 'specificity': 0.3857691830916078, 'f1_score': 0.1850943352972192, 'roc_auc': 0.6819585588692542})


In [21]:
# get all columns except ID
all_features = [col for col in data_fe.columns if col != 'SK_ID_CURR']
train_all = train_data_random[all_features].copy()
val_all = val_data_random[all_features].copy()

log_reg_metrics_all = cross_validate(log_reg, train_all, val_all, cv=5)
print(log_reg_metrics_all)
svc_metrics_all = cross_validate(svm_model, train_all, val_all, cv=5)
print(svc_metrics_all)
lda_metrics_all = cross_validate(lda_model, train_all, val_all, cv=5)
print(lda_metrics_all)


Number of NA in features: 0


c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

({'accuracy': 0.4969690149990537, 'recall': 0.8267241787005724, 'precision': 0.12040053517949685, 'specificity': 0.4679355579047194, 'f1_score': 0.21017596468194202, 'roc_auc': 0.7267029884379774}, array([[ 3.87672047e-02, -2.18576417e-03,  1.74772322e-02,
         1.38212125e-01,  3.82772717e-02,  1.94018205e-01,
         3.38662297e-02,  9.47565529e-02, -1.91403664e-02,
        -3.59719783e-05, -2.42802786e-02,  1.88269029e-02,
        -4.31554646e-01,  8.55271044e-02,  9.03391431e-02,
        -5.11652236e-02, -8.75219656e-02,  3.32863733e-02,
        -5.10044930e-02, -1.01220500e-01,  1.77403957e-01,
         1.67193071e-01, -2.63706066e-02,  5.29664940e-02,
         6.41647033e-02,  9.52216465e-02, -6.41234104e-02,
        -1.99173167e-02,  5.21600173e-03, -2.70731915e-02,
        -1.59800424e-02, -3.66765503e-02, -1.36063446e-01,
        -2.94165154e-02,  2.11030861e-02, -6.00388713e-02,
        -1.10845348e-02, -1.15352148e-02,  2.80251904e-01,
         2.21891127e-01, -1.9583961